# CNN-Transformer Only — Kaggle Pipeline

This notebook trains and evaluates the standalone CNN-Transformer IDS pipeline and produces:
- model checkpoint (`cnn_transformer_ids.pth`)
- Integrated Gradients ranking CSV
- Grad-CAM ranking CSV (+ optional plot)
- SHAP global importance CSV (+ plots)

In [ ]:
# Cell 1: Setup & install
import os, sys, glob, warnings
warnings.filterwarnings('ignore')
os.environ['TORCHDYNAMO_DISABLE'] = '1'

REPO_URL = 'https://github.com/samaraweeramethun-eng/IDS_Interpretability.git'
REPO_DIR = '/kaggle/working/IDS_Interpretability'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

!pip install -q shap joblib psutil
!pip install -q -e cnn-transformer-only

import torch
import cnn_transformer_only
print('✓ cnn_transformer_only:', cnn_transformer_only.__version__)
print('✓ PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✓ GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: Data discovery (auto-detect under /kaggle/input)
import pandas as pd

KAGGLE_DATASET_PATH = ''  # optional: set exact path to a CSV

DATA_PATH = None
if KAGGLE_DATASET_PATH and os.path.exists(KAGGLE_DATASET_PATH):
    DATA_PATH = KAGGLE_DATASET_PATH

if DATA_PATH is None:
    candidates = glob.glob('/kaggle/input/**/cicids*.csv', recursive=True)
    if candidates:
        candidates.sort(key=os.path.getsize, reverse=True)
        DATA_PATH = candidates[0]
        print('Auto-detected:', DATA_PATH)

# Fallback to repo sample if present
if DATA_PATH is None:
    if os.path.exists('data/cicids2017/cicids2017.csv'):
        DATA_PATH = 'data/cicids2017/cicids2017.csv'
    else:
        DATA_PATH = 'data/cicids2017/cicids2017_sample.csv'

# If using /kaggle/input, symlink into repo data path for convenience
repo_data_dir = 'data/cicids2017'
os.makedirs(repo_data_dir, exist_ok=True)
repo_csv = os.path.join(repo_data_dir, 'cicids2017.csv')
if DATA_PATH.startswith('/kaggle/input') and not os.path.exists(repo_csv):
    os.symlink(DATA_PATH, repo_csv)
    DATA_PATH = repo_csv

print('Using:', DATA_PATH)
print('Size (MB):', os.path.getsize(DATA_PATH) / 1024**2)

df_peek = pd.read_csv(DATA_PATH, nrows=5)
label_candidates = [c for c in df_peek.columns if 'label' in c.lower()]
print('Columns:', len(df_peek.columns))
print('Label col:', label_candidates[0] if label_candidates else 'NOT FOUND')

In [ ]:
# Cell 3: Configure CNN-Transformer
from cnn_transformer_only.config import CNNTransformerConfig

USE_SAMPLE = ('sample' in DATA_PATH.lower())

cnn_cfg = CNNTransformerConfig(
    input_path=DATA_PATH,
    output_dir='/kaggle/working/artifacts',
    epochs=25 if not USE_SAMPLE else 5,
    batch_size=1024 if not USE_SAMPLE else 64,
    val_batch_size=2048 if not USE_SAMPLE else 128,
    lr=1.5e-3,
    num_workers=2,
    d_model=192 if not USE_SAMPLE else 64,
    conv_channels=96 if not USE_SAMPLE else 32,
    num_layers=3 if not USE_SAMPLE else 1,
    num_heads=8 if not USE_SAMPLE else 4,
    d_ff=768 if not USE_SAMPLE else 256,
    ig_steps=32 if not USE_SAMPLE else 8,
    ig_samples=512 if not USE_SAMPLE else 128,
    max_train_samples=500_000 if not USE_SAMPLE else 0,
)

print('Mode:', 'SAMPLE' if USE_SAMPLE else 'FULL')
print('Output dir:', cnn_cfg.output_dir)
print('Config:', cnn_cfg.epochs, 'epochs | d_model', cnn_cfg.d_model, '| batch', cnn_cfg.batch_size)

In [ ]:
# Cell 4: Train CNN-Transformer
import gc, time, psutil
from cnn_transformer_only.training.cnn_trainer import train_cnn_transformer

os.makedirs(cnn_cfg.output_dir, exist_ok=True)

ram = psutil.virtual_memory()
print(f'System RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

gc.collect()
t0 = time.time()
cnn_path = train_cnn_transformer(cnn_cfg)
print('Checkpoint:', cnn_path)
print('Elapsed (min):', (time.time()-t0)/60)

In [ ]:
# Cell 5: Print test metrics from checkpoint
ckpt = torch.load(cnn_path, map_location='cpu', weights_only=False)
m = ckpt.get('test_metrics', {})
print('=== Test Set Metrics ===')
for k in ['auc_roc','auc_pr','f1_score','recall','precision','accuracy']:
    print(f'{k}: {m.get(k, None)}')
print('Best threshold:', ckpt.get('best_threshold', 0.5))

In [ ]:
# Cell 6: View IG + Grad-CAM top features
import pandas as pd
OUT = cnn_cfg.output_dir

ig_path = os.path.join(OUT, 'cnn_transformer_integrated_gradients.csv')
cam_path = os.path.join(OUT, 'cnn_transformer_grad_cam.csv')

if os.path.exists(ig_path):
    print('=== Top 15 — Integrated Gradients ===')
    display(pd.read_csv(ig_path).head(15))
else:
    print('Missing:', ig_path)

if os.path.exists(cam_path):
    print('=== Top 15 — Grad-CAM ===')
    display(pd.read_csv(cam_path).head(15))
else:
    print('Missing:', cam_path)

In [ ]:
# Cell 7: Run SHAP (optional, can be slow)
from cnn_transformer_only.interpretability.shap_runner import run_shap

shap_dir = os.path.join(cnn_cfg.output_dir, 'shap')
os.makedirs(shap_dir, exist_ok=True)

SHAP_BG = 2000 if not USE_SAMPLE else 200
SHAP_EVAL = 2000 if not USE_SAMPLE else 200
SHAP_POOL = 150_000 if not USE_SAMPLE else 500

csv = run_shap(
    checkpoint_path=cnn_path,
    data_path=DATA_PATH,
    output_dir=shap_dir,
    background_size=SHAP_BG,
    eval_size=SHAP_EVAL,
    eval_pool=SHAP_POOL,
    chunk_size=256,
)
print('SHAP CSV:', csv)
if os.path.exists(csv):
    display(pd.read_csv(csv).head(15))